In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
groq_api_key = os.getenv("GROQ_API_KEY")


In [4]:
from langchain_groq import ChatGroq
model = ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

d:\Python\GEN AI\Chatbot with langchain\venv2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000178830956C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000017883095DB0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [5]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content = " my name is srimad , i am in btech")])


AIMessage(content="Nice to meet you, Srimad! You're a B.Tech student, that's great. Which branch are you studying in (like CSE, ECE, Mech, etc.) and which year are you in?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 47, 'total_tokens': 94, 'completion_time': 0.087202028, 'completion_tokens_details': None, 'prompt_time': 0.002279945, 'prompt_tokens_details': None, 'queue_time': 0.046449203, 'total_time': 0.089481973}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019caf09-9a29-7ff2-ab32-c32c952219e6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 47, 'output_tokens': 47, 'total_tokens': 94})

In [6]:
from langchain_core.messages import AIMessage
model.invoke(
[
    HumanMessage(content= "my name is srimad , i am in btech"),
    AIMessage(content="Nice to meet you, Srimad. Being a B Tech student can be a challenging yet rewarding experience. Which branch of engineering are you studying? Are you in your first year or have you progressed to a higher semester? "),
    HumanMessage(content = "hey whats my name what do i do")
]

)

AIMessage(content="Your name is Srimad, and you're a B Tech student.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 111, 'total_tokens': 127, 'completion_time': 0.035473774, 'completion_tokens_details': None, 'prompt_time': 0.006419103, 'prompt_tokens_details': None, 'queue_time': 0.060295717, 'total_time': 0.041892877}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019caf09-9ba9-7010-8454-fd7870a4593a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 16, 'total_tokens': 127})

In [7]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [8]:
config = {"configurable":{"session_id":"chat1"}}

In [9]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name srimad and i m in my 2nd year of btech")],
    config=config
)

In [10]:
response.content

"Hello Srimad, nice to meet you. Being in your 2nd year of B.Tech is a great milestone. How's your journey so far? Are you enjoying your course and college life?"

In [11]:
#change the config
config1 = {"configurable": {"session_id": "chat2"}}

response = with_message_history.invoke(
    [HumanMessage(content="whats my name")],
    config=config1
)

print(response.content)


I don't have any information about your name as our conversation has just started. I'm happy to chat with you and help with any questions you have, but I don't have any prior knowledge about you. If you'd like to share your name, I'd be happy to learn it!


In [12]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is srimad")],
    config=config1
)
response.content

"Nice to meet you, Srimad. It's a unique and interesting name. I don't have any information about the origin or significance of your name, but I'd be happy to learn more about it if you'd like to share. What's the story behind your name, if you don't mind me asking?"

In [13]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is Srimad.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [14]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant , ans all my questions in  and help me "

        ),
        MessagesPlaceholder(variable_name="message"),
        
    ]
)
chain = prompt|model

In [15]:
chain.invoke({
    "message": [HumanMessage(content="my name is srimad")]
})



AIMessage(content="Nice to meet you, Srimad! I'm here to help you with any questions or information you need. What's on your mind, and how can I assist you today?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 38, 'prompt_tokens': 56, 'total_tokens': 94, 'completion_time': 0.051299595, 'completion_tokens_details': None, 'prompt_time': 0.002678347, 'prompt_tokens_details': None, 'queue_time': 0.045129233, 'total_time': 0.053977942}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019caf09-a4a7-7a92-9cd5-400d6df1dfb3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 56, 'output_tokens': 38, 'total_tokens': 94})

In [16]:
with_message_history= RunnableWithMessageHistory(chain,get_session_history)

In [17]:
config = {"configurable": {"session_id": "chat3"}}

response = with_message_history.invoke(
    [HumanMessage(content="hi my name is srimad")],
    config=config
)

print(response)

content="Nice to meet you, Srimad. I'm happy to assist you with any questions or topics you'd like to discuss. How's your day going so far?" additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 57, 'total_tokens': 92, 'completion_time': 0.047741377, 'completion_tokens_details': None, 'prompt_time': 0.002784594, 'prompt_tokens_details': None, 'queue_time': 0.046536136, 'total_time': 0.050525971}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019caf09-a854-7113-a7b4-46699258575f-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 57, 'output_tokens': 35, 'total_tokens': 92}


In [18]:
#adding more complexity
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant , ans all my questions{language} and help me "

        ),
        MessagesPlaceholder(variable_name="message"),
        
    ]
)
chain = prompt|model

In [19]:
response = chain.invoke({"message":[HumanMessage(content = "Hi my name is srimad")],"language" : "Hindi"})

In [20]:
response.content

'नमस्ते स्रीमद जी! (Namaste Srimad ji!) कैसे जा रहे हैं? (How are you?) मैं आपकी मदद करने के लिए यहाँ हूँ।'

Now wrapping chain in a message history class

In [21]:
with_message_history= RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="message"
)

In [22]:
config = {"configurable": {"session_id": "chat4"}}

response = with_message_history.invoke(
    {
        "message":[HumanMessage(content= " hey my name is srimad nayak")],"language":"hindi"
    },
    config = config
)
response.content

'नमस्ते श्रीमद नायक जी (Namaste Srimad Nayak Ji)\n\nकृपया पूछें कि क्या मैं आपकी मदद कर सकता हूँ।'

In [23]:
response = with_message_history.invoke(
    {
        "message":[HumanMessage(content= " whats my  name")],"language":"hindi"
    },
    config = config
)
response.content

'आपका नाम है - श्रीमद नायक'

Managing the conversation history,
trim message =A utility function that reduces chat history based on token limits.


In [26]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer = trim_messages(

    max_tokens=75,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)

messages = [
    SystemMessage(content="you are a good assistant"),
    HumanMessage(content="hi i m srimad"),
    AIMessage(content="hello"),
    HumanMessage(content="i m from xim"),
    AIMessage(content="oh nice"),
    HumanMessage(content="whats ur name"),
    AIMessage(content="I am an ai assistant"),
    HumanMessage(content="good"),
    AIMessage(content="yup"),
    HumanMessage(content="i am human"),
    AIMessage(content="i am an LLm"),
    HumanMessage(content="nice"),
    AIMessage(content="yeah"),


]
trimmer.invoke(messages)

[SystemMessage(content='you are a good assistant', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='hi i m srimad', additional_kwargs={}, response_metadata={}),
 AIMessage(content='hello', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='i m from xim', additional_kwargs={}, response_metadata={}),
 AIMessage(content='oh nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats ur name', additional_kwargs={}, response_metadata={}),
 AIMessage(content='I am an ai assistant', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='good', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yup', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='i am human', additional_kwargs={}, response_metadata={}),
 AIMessage(content='i am an LLm', addi